# resolve_time

Hier geht es darum wie gut das LLM mit einer Referenzzeit umgehen und gestern, letzte Woche usw interpretieren kann. Verglichen wird eine Zeitauflösung mit und ohne Tools sowohl für die aktuelle Zeit als auch für eine Referenzzeit.

Es ist entweder möglich die Testdaten immer wieder neu für die letzten Tage zu generieren und darauf zu achten, dass diese numerisch gleich bleiben, aber immer vom aktuellen Tag als Referenz rückwirkend generiert werden. Allerdings unterscheiden sich dann Interpretationen zum Beispiel von "letzte Woche" je nach aktuellem Wochentag.

Stabiler und schöner ist es ein fixes Testdatenset zu erstellen das man auch nicht mehr verändert und dem LLM einen fixen Referenzzeitpunkt zu geben von dem es ausgeht, dass dieser heute ist.

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

def call_llm(user_input: str, systeminformation: str, client=client, messages=[]):
    response = client.chat.completions.create(
        #model="gpt-5.4",
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": systeminformation},
            *messages,
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [18]:
SYSTEMINFORMATION = """
Heute ist der 2026-01-22.

Du bist ein Planer für Tool-Aufrufe.

Tool:
get_consumption(start, end)

Regeln:
- Konvertiere relative Zeitangaben in konkrete Daten
- Verwende Format YYYY-MM-DD
- Zeitintervalle sind im Format [start, end) definiert (inkl. start, exkl. end)
- Wenn ein einzelner Tag abgefragt wird: setze start auf diesen Tag, setze end auf den folgenden Tag
- Wenn eine Zeitangabe mehrdeutig ist, treffe keine Annahme, sondern frage nach.

Gib NUR JSON zurück:

{
  "plan": [...],
  "reasoning": "..."
}
"""

user_input = "Wie ist mein Verbrauch der ersten 3 Tage der Woche vor dem 10. Januar?"
response = call_llm(user_input, SYSTEMINFORMATION)
print(response)

{"plan":[{"tool":"get_consumption","start":"2026-01-05","end":"2026-01-08"}],"reasoning":"Der 10. Januar 2026 ist ein Samstag. Die Woche davor ist die Kalenderwoche vom 2026-01-05 bis 2026-01-11. Die ersten 3 Tage dieser Woche sind Montag bis Mittwoch, also 2026-01-05 bis 2026-01-08 im Intervall [start, end)."}
